In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.append(os.path.abspath('../'))

from tagging_system import automate_tagging as auto_tag

In [4]:
books = pd.read_csv('../data/data_with_author_and_awards.csv',
dtype = {
    'isbn' : 'str', # do this explicitly to avoide getting a warning by the interpreter
    'author_birthyear' : 'Int64', # we have to explicitly do this to avoid pandas implicitly casting as float
    'title_id' : 'Int64'
},
)
books = books.dropna()

In [5]:
tags = auto_tag.load_tags()
transformer = auto_tag.load_transformer()
tags_encoded = auto_tag.encode_tags()
tags_map = auto_tag.tag_dictionary()

In [6]:
books = auto_tag.encode_books(books, transformer) # takes a while

In [8]:
books = books[books.release_year >= 1971] # leaves around 20k rows

In [9]:
# manually train test split our data

books_train = books[books.release_year <= 2014]
books_test = books[books.release_year <= 2014]

books_tt = books_train[books_train.release_year <= 2005]
books_val = books_train[books_train.release_year > 2005]

In [10]:
books_tt.head()

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,...,locus,month_of_publication,Author_Age_at_Publication,Hugo_Awards_Previously,Locus_Awards_Previously,Hugo_Nominee_Before,Locus_Nominee_Before,author_birthplace_country,author_birthplace_continent,encoded_synopsis
4548,2009247,The Hawkstone,Jay Williams,1971,1971-10-01,"Henry Z. Walck, Inc.",1914,"Buffalo, New York, USA",0809830965,"""A young New Englander is unaware of the stran...",...,False,10,57,0,0,False,False,USA,Central/North America,"[-0.036100663, 0.10305314, 0.032459974, 0.0384..."
4552,1997735,The Seal-Singing,Rosemary Harris,1971,1971-00-00,Faber and Faber,1923,"London, England, UK",0571096395,"""While spending the summer on a Scottish islan...",...,False,Unknown,48,0,0,False,False,UK,Europe,"[-0.07222478, 0.048827797, -0.00031475644, 0.0..."
4553,449,Exiled from Earth,Ben Bova,1971,1971-04-00,Berkley Books,1932,"Philadelphia, Pennsylvania, USA",0425045250,3 science fiction-romaner.\n\n,...,False,04,39,0,0,False,False,USA,Central/North America,"[-0.08211492, -0.05334223, -0.0065231794, 0.01..."
4557,13636,The Exorcist,William Peter Blatty,1971,1971-00-00,HarperPaperbacks,1928,"New York City, New York, USA",0061007226,The Exorcist is a 1971 horror novel by America...,...,False,Unknown,43,0,0,False,False,USA,Central/North America,"[-0.0025305643, 0.010534443, -0.02623904, 0.05..."
4561,13885,The Vampire Curse,Daoma Winston,1971,1971-00-00,Severn House,1922,"Washington, District of Columbia, USA",0727844490,Teena Halliday had heard rumours that Jeremy R...,...,False,Unknown,49,0,0,False,False,USA,Central/North America,"[-0.027311608, 0.05795092, 0.008173795, 0.0292..."


Here we'll just use the book synopsis encodings to form some simple models.

In [11]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

In [12]:
models = {
    'KNN' : KNeighborsClassifier(),
    'RFC' : RandomForestClassifier(),
    'AdaBoost' : AdaBoostClassifier(),
    'GradBoost' : GradientBoostingClassifier(),
    'XGB' : XGBClassifier()
}

In [ ]:
encoded_synopsis_tt = books_tt.encoded_synopsis.explode().values.astype(float).reshape(-1, 384) # these are the hacks I need to plug the vectors in as features
encoded_synopsis_val = books_val.encoded_synopsis.explode().values.astype(float).reshape(-1, 384) # Do this first for performance reasons

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score

scores = dict()
preds = dict()
# do the locus
# this training step will take a while
for m in models:
    models[m].fit(encoded_synopsis_tt, books_tt.locus)
    pred = models[m].predict(encoded_synopsis_val) 
    preds[m] = pred

    accuracy = accuracy_score(books_val.locus, preds[m])
    f1 = f1_score(books_val.locus, preds[m])
    precision = precision_score(books_val.locus, preds[m])

    scores[m] = (accuracy, f1, precision)

scores


/home/tiger/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'KNN': (0.9509352959214965, 0.08571428571428572, 0.16304347826086957),
 'RFC': (0.9604415823367065, 0.0, 0.0),
 'AdaBoost': (0.9601349279362159, 0.0, 0.0),
 'GradBoost': (0.9599816007359706, 0.0, 0.0),
 'XGB': (0.9590616375344986, 0.02909090909090909, 0.23529411764705882)}

In [49]:
from sklearn.metrics import confusion_matrix

for m in models:
    print(m, confusion_matrix(books_val.locus, preds[m], normalize='all'))

KNN [[0.94863539 0.01180619]
 [0.03725851 0.00229991]]
RFC [[0.96044158 0.        ]
 [0.03955842 0.        ]]
AdaBoost [[9.60134928e-01 3.06654400e-04]
 [3.95584177e-02 0.00000000e+00]]
GradBoost [[9.59981601e-01 4.59981601e-04]
 [3.95584177e-02 0.00000000e+00]]
XGB [[9.58448329e-01 1.99325360e-03]
 [3.89451089e-02 6.13308801e-04]]
